# Day 36: Random Forest Hyperparameter Tuning
**Author:** Sahil-K-Y  
**Phase:** 3 - Tree Models & SVM  
**Date:** Day 036

---

## 1. Essential Hyperparameters of Random Forest

Hyperparameter tuning is crucial to regularize tree growth, balance bias-variance, and optimize performance. Here are the six core hyperparameters in Scikit-Learn's `RandomForestClassifier`:

1. **`n_estimators` (default=100):** The number of trees in the forest. Generally, more trees improve performance and stability, but trigger diminishing returns and higher computational overhead.
2. **`max_depth` (default=None):** The maximum depth of each tree. Constraining depth (e.g., 5 to 15) restricts tree growth, which reduces variance (overfitting) at the cost of slight bias.
3. **`min_samples_split` (default=2):** The minimum number of samples required to split an internal node. Raising this value prevents trees from capturing highly specific, noisy splits.
4. **`min_samples_leaf` (default=1):** The minimum number of samples required at a leaf node. Setting this higher smooths out decision boundaries and regularizes predictions.
5. **`max_features` (default="sqrt"):** The number of features to consider when looking for the best split. Lower values create more diverse trees and decrease overfitting.
6. **`bootstrap` (default=True):** Whether bootstrap samples are used when building trees. If False, the entire dataset is used to build each tree (which is not recommended since bootstrap variance reduction is lost).


## 2. Tuning Strategies: Grid Search vs. Randomized Search

* **GridSearchCV:** Performs an exhaustive grid search over all combinations of parameter values.
  * *Pros:* Guaranteed to find the absolute best combination in the specified grid.
  * *Cons:* Highly expensive computationally. Trying 4 values for 5 parameters requires $4^5 = 1024$ fits. Multiplied by 5-fold CV, that's 5120 training runs.
* **RandomizedSearchCV:** Evaluates a fixed number of random combinations sampled from defined distributions.
  * *Pros:* Extremely fast; lets you explore wider parameter ranges; achieves comparable results to Grid Search in a fraction of the time.
  * *Cons:* Not guaranteed to find the absolute global maximum in the grid.


## Exercise 1: GridSearchCV — `n_estimators` and `max_depth`

We will use the **Breast Cancer Dataset** to perform hyperparameter tuning. First, we will exhaustively search the grid of the two most critical parameters.

### Step 1.1: Load Breast Cancer Dataset
Let's load the data.


In [ ]:
from sklearn.datasets import load_breast_cancer
import pandas as pd

bc = load_breast_cancer()
X, y = bc.data, bc.target
feature_names, class_names = bc.feature_names, bc.target_names

print(f"Dataset shape: {X.shape}")
print(f"Classes: {class_names}")


### Step 1.2: Train-Test Split
We split the data.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"Training samples: {X_train.shape[0]} | Testing samples: {X_test.shape[0]}")


### Step 1.3: Train Baseline Model
Let's train a default Random Forest model to establish a baseline accuracy.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

baseline_rf = RandomForestClassifier(random_state=42)
baseline_rf.fit(X_train, y_train)
y_pred_base = baseline_rf.predict(X_test)
print(f"Baseline Accuracy: {accuracy_score(y_test, y_pred_base):.4%}")


### Step 1.4: GridSearchCV Search
Let's optimize `n_estimators` and `max_depth` with GridSearchCV.


In [ ]:
from sklearn.model_selection import GridSearchCV
import time

param_grid_1 = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [3, 5, 10, 15, None]
}

t0 = time.time()
grid_search_1 = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid_1,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid_search_1.fit(X_train, y_train)
t1 = time.time()

print(f"GridSearchCV completed in {t1-t0:.2f} seconds.")
print(f"Best Parameters: {grid_search_1.best_params_}")
print(f"Best CV Accuracy: {grid_search_1.best_score_:.4%}")


### Step 1.5: Heatmap of Grid Results
Let's pivot the result grid and display it visually.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

results_1 = pd.DataFrame(grid_search_1.cv_results_)
pivot_1 = results_1.pivot(index='param_max_depth', columns='param_n_estimators', values='mean_test_score')
pivot_1.index = [str(idx) for idx in pivot_1.index]

sns.heatmap(pivot_1, annot=True, fmt='.4%', cmap='YlOrRd')
plt.title('CV Accuracy: max_depth vs. n_estimators', fontsize=14, fontweight='bold')
plt.xlabel('Number of Estimators')
plt.ylabel('Maximum Depth')
plt.show()


## Exercise 2: GridSearchCV — Regularization Parameters

Now, let's look at `min_samples_split` and `min_samples_leaf` while holding tree depth and count constant.

### Step 2.1: Run Regularization GridSearchCV
We define the parameter grid and fit it.


In [ ]:
param_grid_2 = {
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8]
}

t0 = time.time()
grid_search_2 = GridSearchCV(
    estimator=RandomForestClassifier(n_estimators=100, random_state=42),
    param_grid=param_grid_2,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid_search_2.fit(X_train, y_train)
t1 = time.time()

print(f"GridSearchCV completed in {t1-t0:.2f} seconds.")
print(f"Best Parameters: {grid_search_2.best_params_}")
print(f"Best CV Accuracy: {grid_search_2.best_score_:.4%}")


### Step 2.2: Heatmap of Regularization Search
Let's plot the regularization results grid.


In [ ]:
results_2 = pd.DataFrame(grid_search_2.cv_results_)
pivot_2 = results_2.pivot(index='param_min_samples_split', columns='param_min_samples_leaf', values='mean_test_score')

sns.heatmap(pivot_2, annot=True, fmt='.4%', cmap='GnBu')
plt.title('CV Accuracy: min_samples_split vs. min_samples_leaf', fontsize=14, fontweight='bold')
plt.xlabel('Minimum Samples Leaf')
plt.ylabel('Minimum Samples Split')
plt.show()


## Exercise 3: RandomizedSearchCV — Full Tuning Sweep

Let's tune all six hyperparameters at once to demonstrate the speed and efficacy of `RandomizedSearchCV`.

### Step 3.1: Execute RandomizedSearchCV
We'll define parameter distributions using Scipy's `sp_randint`.


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint as sp_randint

param_dist = {
    'n_estimators': sp_randint(50, 400),
    'max_depth': [3, 5, 10, 15, 20, None],
    'min_samples_split': sp_randint(2, 30),
    'min_samples_leaf': sp_randint(1, 15),
    'max_features': ['sqrt', 'log2', 0.2, 0.4, 0.6],
    'bootstrap': [True, False]
}

t0 = time.time()
rand_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=100,
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1
)
rand_search.fit(X_train, y_train)
t1 = time.time()

print(f"RandomizedSearchCV completed in {t1-t0:.2f} seconds.")
print(f"Best Parameters: {rand_search.best_params_}")
print(f"Best CV Accuracy: {rand_search.best_score_:.4%}")


## Exercise 4: Learning Curves Analysis

A learning curve helps diagnose if the tuned model suffers from high bias (underfitting) or high variance (overfitting).

### Step 4.1: Compute Learning Curves
We'll import `learning_curve` and generate score trends over different training set sizes.


In [ ]:
from sklearn.model_selection import learning_curve
import numpy as np

best_rf = rand_search.best_estimator_

train_sizes, train_scores, cv_scores = learning_curve(
    estimator=best_rf,
    X=X_train,
    y=y_train,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=5,
    n_jobs=-1,
    random_state=42
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
cv_mean = np.mean(cv_scores, axis=1)
cv_std = np.std(cv_scores, axis=1)
print("Learning curve data computed.")


### Step 4.2: Plot the Curves
Let's render the training score vs cross-validation score.


In [ ]:
plt.plot(train_sizes, train_mean, 'o-', color='blue', label='Training Score')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='blue')
plt.plot(train_sizes, cv_mean, 's-', color='green', label='Cross-validation Score')
plt.fill_between(train_sizes, cv_mean - cv_std, cv_mean + cv_std, alpha=0.15, color='green')

plt.title('Learning Curve (Tuned Random Forest)', fontsize=14, fontweight='bold')
plt.xlabel('Training Set Size')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()


## Exercise 5: Final Evaluation on Hold-Out Test Set

Finally, let's verify if the hyperparameter tuning yields a performance improvement on unseen data.

### Step 5.1: Make Predictions on Test Set
Let's calculate final accuracy for both baseline and tuned models.


In [ ]:
y_pred_tuned = best_rf.predict(X_test)
tuned_acc = accuracy_score(y_test, y_pred_tuned)

print("===== TEST SET COMPARISON =====")
print(f"Baseline Accuracy: {accuracy_score(y_test, y_pred_base):.4%}")
print(f"Tuned Model Accuracy: {tuned_acc:.4%}")
print(f"Absolute Lift: {tuned_acc - accuracy_score(y_test, y_pred_base):+.4%}")


### Step 5.2: Confusion Matrix & Classification Report
Let's visualize target label metrics and correct predictions.


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(y_test, y_pred_tuned)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix (Tuned Random Forest)', fontsize=12, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

print("\nClassification Report:")
print(classification_report(y_test, y_pred_tuned, target_names=class_names))
